# 02 Analysis
Answers the question: between ONS years 2019 and 2024, did housing become less affordable in England's councils, and was that driven by prices or by pay?

**What this notebook is for:** It's responsible for comparing the first and last years for each council, splitting the change into price and pay, testing London against the rest, and writing the final answer with its limitations.

In [3]:
import sys
sys.path.append("..")  # lets the notebook import from the pipeline package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pipeline.db import get_engine

engine = get_engine()
pd.read_sql("select count(*) from core.fact_sale", engine)

,count
0,5633489


In [4]:
cy = pd.read_sql("select * from mart.council_year where not low_volume", engine)
first, last = cy["ons_year"].min(), cy["ons_year"].max()
wide = cy[cy.ons_year == first].merge(
    cy[cy.ons_year == last], on=["council_code", "council_name"], suffixes=("_0", "_1")
)
all_councils = pd.read_sql("select count(distinct council_code) as n from mart.council_year", engine).n[0]
print(f"Comparing {first} with {last}: {len(wide)} councils kept, {all_councils - len(wide)} left out")

Comparing 2019 with 2024: 294 councils kept, 2 left out


## One row per council
**Saw:** comparing 2019 with 2024, 294 councils kept and 2 left out because they were low volume (the City of London and the Isles of Scilly).

_______________________________________________________________________________________________________________________________

## Ranking the change
**Expect:** the biggest rises in affordability ratio in places where prices grew fast, and some London boroughs becoming *more* affordable, because London prices were flat while pay rose.

In [5]:
wide["ratio_change"] = wide.ratio_1 - wide.ratio_0
columns = ["council_name", "ratio_0", "ratio_1", "ratio_change"]
print("Became less affordable:")
print(wide.nlargest(10, "ratio_change")[columns].round(2).to_string(index=False))
print("\nBecame more affordable:")
print(wide.nsmallest(10, "ratio_change")[columns].round(2).to_string(index=False))

Became less affordable:
         council_name  ratio_0  ratio_1  ratio_change
                Lewes     9.74    10.93          1.19
           Chichester    10.50    11.67          1.17
           Manchester     6.76     7.87          1.11
               Maldon     9.13    10.22          1.10
 South Cambridgeshire     9.04    10.08          1.04
              Gedling     6.17     7.18          1.01
           Canterbury     8.67     9.64          0.98
                Luton     8.12     9.05          0.94
Hinckley and Bosworth     6.96     7.86          0.90
              Ashford     8.92     9.81          0.89

Became more affordable:
          council_name  ratio_0  ratio_1  ratio_change
Kensington and Chelsea    27.64    23.82         -3.83
               Lambeth    14.36    11.17         -3.20
               Hackney    16.04    12.98         -3.06
           Mole Valley    14.89    11.97         -2.92
          Three Rivers    14.29    11.49         -2.80
              Lewisham    1

**Saw:** the biggest rise was Lewes (from 9.74 to 10.93), and the biggest fall was Kensington and Chelsea (from 27.64 to 23.82). 111 councils became less affordable and 183 more affordable. The less affordable end is mostly commuter-belt and coastal South East (Lewes, Chichester, Maldon, South Cambridgeshire) plus Manchester. The more affordable end is led by London boroughs (Kensington and Chelsea, Lambeth, Hackney).

_______________________________________________________________________________________________________________________________

## Prices or pay?
**Expect:** in most councils both prices and pay went up, so the question is which went up faster. Splitting the change in log ratio gives a price part and a pay part that add up exactly.

In [7]:
wide["d_ratio"] = np.log(wide.ratio_1) - np.log(wide.ratio_0)
wide["d_price"] = np.log(wide.median_price_1) - np.log(wide.median_price_0)
wide["d_earn"] = np.log(wide.median_earnings_1.astype(float)) - np.log(wide.median_earnings_0.astype(float))
assert np.allclose(wide.d_ratio, wide.d_price - wide.d_earn)

wide["group"] = np.select(
    [
        (wide.d_price > 0) & (wide.d_earn > 0),
        (wide.d_price > 0) & (wide.d_earn <= 0),
        (wide.d_price <= 0) & (wide.d_earn > 0),
    ],
    ["prices and pay both up", "prices up, pay flat or down", "prices flat or down, pay up"],
    default="both flat or down",
)
wide["group"].value_counts()

group
prices and pay both up         292
prices flat or down, pay up      2
Name: count, dtype: int64

**Saw:** in 292 councils prices and pay both rose, in 0 prices rose while pay was flat or fell, and in 2 prices fell while pay rose. The split adds up exactly (the assert passed). Within the 292, prices outpaced pay in 111 and pay outpaced prices in 181, which is why most councils got slightly more affordable.

_______________________________________________________________________________________________________________________________

## London against the rest
**Expect:** London's ratio change to be smaller than the rest of England's, because London prices barely moved after 2019 while prices elsewhere rose.

In [8]:
from scipy.stats import mannwhitneyu

is_london = wide.council_code.str.startswith("E09")
london, rest = wide[is_london].d_ratio, wide[~is_london].d_ratio
result = mannwhitneyu(london, rest)
print(f"London median change: {london.median():.3f} ({len(london)} boroughs)")
print(f"Rest median change:   {rest.median():.3f} ({len(rest)} councils)")
print(f"Mann-Whitney U p-value: {result.pvalue:.4f}")

London median change: -0.093 (32 boroughs)
Rest median change:   -0.012 (262 councils)
Mann-Whitney U p-value: 0.0000


**Saw:** London's median change in log ratio was −0.093 (about 9% more affordable) against −0.012 (about 1%) for the rest of England, with a Mann-Whitney p-value of 0.0000001. That's strong evidence that London behaved differently, since p is far below 0.05. London prices barely moved while pay rose, so 29 of the 32 boroughs became more affordable.

_______________________________________________________________________________________________________________________________

## New builds
**Expect:** a weak link at most. More new builds might mean prices rose more, or it might not; either way this shows association, not cause.

In [9]:
from scipy.stats import spearmanr

rho, p = spearmanr(wide.new_build_share_1, wide.d_ratio)
print(f"Spearman rho = {rho:.2f}, p = {p:.4f}")

Spearman rho = 0.10, p = 0.0863


**Saw:** Spearman's rho measures whether two things rise together, using ranks rather than raw values, so a few extreme councils can't dominate.

Spearman's rho is 0.10 (p = 0.086), a very weak positive association that isn't statistically significant (p is above 0.05). There's no real evidence that councils with more new builds changed differently, and it wouldn't show cause even if there were.

_______________________________________________________________________________________________________________________________

## Answer
Between 2019 and 2024, affordability worsened in 111 of 294 councils, and in every one it was because prices rose faster than pay, not because pay fell. In the other 183, pay outpaced prices, and London improved most (median about 9% more affordable, against about 1% elsewhere).

Limitations:
- Sale prices, not rents or mortgage costs, so the rise in interest rates since 2022 isn't captured.
- Earnings are for full-time workers only.
- Medians are affected by the property mix (the composition effect).
- 2 low-volume councils excluded (the City of London and the Isles of Scilly).